In [7]:
# Les fonctions communes sont dans fonctions.py.
import fonctions as fct


In [8]:
def mise_a_jour_bfgs(H, s, y):
    """Ameliore l'estimation inverse de la courbure utilisee par BFGS."""
    n = len(s)
    rho = 1.0 / fct.produit_scalaire(y, s)
    H_y = fct.produit_matrice_vecteur(H, y)
    coefficient = rho + rho ** 2 * fct.produit_scalaire(y, H_y)

    nouvelle_H = []
    for i in range(n):
        ligne = []
        for j in range(n):
            correction = rho * (s[i] * H_y[j] + H_y[i] * s[j])
            ajout = coefficient * s[i] * s[j]
            ligne.append(H[i][j] - correction + ajout)
        nouvelle_H.append(ligne)

    return nouvelle_H


def quasi_newton_bfgs(f, x0, grad=None, eps=1e-6, max_iter=1000):
    """Minimise f avec BFGS et renvoie le point trouve et l'historique."""
    # Si le gradient n'est pas fourni, on l'estime a partir de f.
    if grad is None:
        grad = lambda x: fct.gradient_numerique(f, x)

    x = x0[:]  # On garde x0 intact.
    inverse_courbure = fct.matrice_identite(len(x))
    historique = []

    for k in range(max_iter + 1):
        gradient = grad(x)
        longueur_gradient = fct.norme(gradient)
        historique.append((k, f(x), longueur_gradient))

        if longueur_gradient <= eps or k == max_iter:
            break

        # BFGS utilise la courbure estimee pour choisir une direction.
        direction = fct.produit_matrice_vecteur(inverse_courbure, gradient)
        direction = fct.produit_par_scalaire(-1.0, direction)
        pas = fct.backtracking(f, x, gradient, direction)
        nouveau_x = fct.somme(x, fct.produit_par_scalaire(pas, direction))
        nouveau_gradient = grad(nouveau_x)

        # s est le changement de position ; y est le changement de gradient.
        s = fct.difference(nouveau_x, x)
        y = fct.difference(nouveau_gradient, gradient)

        # On met a jour la courbure seulement si les donnees sont fiables.
        if fct.produit_scalaire(y, s) > 1e-12:
            inverse_courbure = mise_a_jour_bfgs(inverse_courbure, s, y)

        x = nouveau_x

    return x, historique


In [9]:
# Verifie une propriete de la mise a jour BFGS sur des vecteurs simples.
s_test = [0.5, -1.0, 0.25]
y_test = [2.0, 0.3, 1.5]
matrice_initiale = fct.matrice_identite(3)
matrice_suivante = mise_a_jour_bfgs(matrice_initiale, s_test, y_test)

produit_Hy = fct.produit_matrice_vecteur(matrice_suivante, y_test)
erreur = fct.norme(fct.difference(produit_Hy, s_test))
erreur_symetrie = abs(matrice_suivante[0][1] - matrice_suivante[1][0])

print(f"y * s = {fct.produit_scalaire(y_test, s_test):.3f}")
print(f"Erreur de la condition secante : {erreur:.2e}")
print(f"Erreur de symetrie             : {erreur_symetrie:.2e}")


y * s = 1.075
Erreur de la condition secante : 5.44e-16
Erreur de symetrie             : 0.00e+00


In [10]:
EPS = 1e-6

# Teste BFGS sur une fonction quadratique.
x, historique = quasi_newton_bfgs(
    fct.f_quadratique, fct.X0_QUAD,
    grad=fct.grad_quadratique, eps=EPS
)
fct.afficher_historique(historique, pas_affichage=1)
print()
fct.resume("BFGS - quadratique", x, historique, fct.X_EXACT_QUAD, EPS)


       k             f(x_k)    ||grad f(x_k)||
       0   5.5500000000e+01          1.766e+01
       1   1.4000000000e+01          1.030e+01
       2   7.3141032522e+00          9.233e+00
       3   7.3399108946e-02          3.470e+00
       4  -2.1566697812e+00          1.021e+00
       5  -2.3886078239e+00          3.457e-02
       6  -2.3888802723e+00          6.811e-03
       7  -2.3888888888e+00          2.169e-05
       8  -2.3888888889e+00          3.246e-07

--- BFGS - quadratique ---
Statut        : convergé
Itérations    : 8
x trouvé      : [0.22222225, 0.11111119, 1.44444445]
x exact       : [0.22222222222222224, 0.11111111111111104, 1.4444444444444446]
||x - x*||    : 7.938e-08
f(x)          : -2.3888888889e+00
||grad f(x)|| : 3.246e-07


In [11]:
# Teste BFGS sur Rosenbrock, dont la vallee est plus difficile a suivre.
x, historique = quasi_newton_bfgs(
    fct.f_rosenbrock, fct.X0_ROSEN,
    grad=fct.grad_rosenbrock, eps=EPS, max_iter=1000
)
fct.afficher_historique(historique)
print()
fct.resume("BFGS - Rosenbrock", x, historique, fct.X_EXACT_ROSEN, EPS)


       k             f(x_k)    ||grad f(x_k)||
       0   5.0820000000e+02          9.313e+02
       3   1.2142201318e+02          2.466e+02
       6   3.3183160695e+00          2.429e+01
       9   1.8638355623e+00          2.125e+00
      12   1.5630188421e+00          1.273e+01
      15   9.9721004183e-01          6.697e+00
      18   8.3717797851e-01          5.903e+00
      21   3.3662896718e-01          7.977e+00
      24   1.1648924431e-01          4.047e+00
      27   2.6901121932e-02          3.510e+00
      30   2.3368202135e-03          5.953e-01
      33   7.3630296049e-06          5.894e-02
      36   6.8438661994e-12          1.198e-04
      38   4.7710568169e-16          8.611e-07

--- BFGS - Rosenbrock ---
Statut        : convergé
Itérations    : 38
x trouvé      : [0.99999999, 0.99999999, 0.99999998]
x exact       : [1.0, 1.0, 1.0]
||x - x*||    : 2.168e-08
f(x)          : 4.7710568169e-16
||grad f(x)|| : 8.611e-07


In [12]:
# Sans argument grad, BFGS estime le gradient a partir de la fonction.
x, historique = quasi_newton_bfgs(
    fct.f_quadratique, fct.X0_QUAD, grad=None, eps=EPS
)
fct.resume("BFGS - gradient numerique", x, historique, fct.X_EXACT_QUAD, EPS)


--- BFGS - gradient numerique ---
Statut        : convergé
Itérations    : 8
x trouvé      : [0.22222225, 0.11111119, 1.44444445]
x exact       : [0.22222222222222224, 0.11111111111111104, 1.4444444444444446]
||x - x*||    : 7.923e-08
f(x)          : -2.3888888889e+00
||grad f(x)|| : 3.243e-07
